# Отчет по пролонгациям аккаунт-менеджеров за 2023 год

M1 (пролонгация в первый месяц) - отношение суммы отгрузки проектов, пролонгированных в первый месяц после завершения, к сумме отгрузки последнего месяца реализации всех проектов, завершившихся в предыдущем месяце.

M2 (пролонгация во второй месяц) - отношение суммы отгрузки проектов, пролонгированных во второй месяц, к сумме отгрузки последнего месяца тех проектов, которые не были пролонгированы в первый месяц.


> Импортируем библиотеки

In [84]:
import pandas as pd
import numpy as np

> Подгружаем dataset

In [85]:
fin = pd.read_csv(r'c:\Users\dimav\Downloads\financial_data.csv')
prol = pd.read_csv(r'c:\Users\dimav\Downloads\prolongations.csv')

print('financial_data:', fin.shape)
print('prolongations :', prol.shape)
display(fin.head())
display(prol.head())

financial_data: (451, 19)
prolongations : (477, 3)


,id,Причина дубля,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account
0,42,NaN,"36 220,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
1,657,первая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
2,657,вторая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
3,594,NaN,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
4,665,NaN,"10 000,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович


,id,month,AM
0,42,ноябрь 2022,Васильев Артем Александрович
1,453,ноябрь 2022,Васильев Артем Александрович
2,548,ноябрь 2022,Михайлов Андрей Сергеевич
3,87,ноябрь 2022,Соколова Анастасия Викторовна
4,429,ноябрь 2022,Соколова Анастасия Викторовна


## Подготовка данных
Что надо исправить:
- суммы в financial_data.csv записаны в формате (36 220,00)
- встречаются значения стоп или пропущенные, которые трактуем как отсутствие отгрузки
- для расчета нужен единый формат даты Timestamp('YYYY-MM-01')

Сделаем 3 функции для форматирования значений: 
- `parse_money(x)` - парсинг денежных значений
- `parse_month_label(s)` - парсинг месячных меток

In [86]:
month_cols = [c for c in fin.columns if c not in ['id', 'Причина дубля', 'Account']]

months = ['январь','февраль','март','апрель','май','июнь',
             'июль','август','сентябрь','октябрь','ноябрь','декабрь']
months_title = ['Январь','Февраль','Март','Апрель','Май','Июнь',
                   'Июль','Август','Сентябрь','Октябрь','Ноябрь','Декабрь']


def parse_money(x):
    if pd.isna(x):
        return 0.0
    s = str(x).strip().lower().replace('\\xa0', '').replace(' ', '')
    if s in ('', 'nan', 'none', '-', 'стоп', 'end'):
        return 0.0
    s = s.replace(',', '.')
    try:
        return float(s)
    except:
        return 0.0
    

def parse_month_label(s):
    s = str(s).strip().lower()
    month_name, year = s.split()
    return pd.Timestamp(year=int(year), month=months.index(month_name) + 1, day=1)


> Применяем функции

In [87]:
for c in month_cols:
    fin[c] = fin[c].map(parse_money)

col_map = {col: parse_month_label(col.lower()) for col in month_cols}
col_by_ts = {v: k for k, v in col_map.items()}

print('Колонки месяцев:', month_cols)

Колонки месяцев: ['Ноябрь 2022', 'Декабрь 2022', 'Январь 2023', 'Февраль 2023', 'Март 2023', 'Апрель 2023', 'Май 2023', 'Июнь 2023', 'Июль 2023', 'Август 2023', 'Сентябрь 2023', 'Октябрь 2023', 'Ноябрь 2023', 'Декабрь 2023', 'Январь 2024', 'Февраль 2024']


## Очистка дублей

- в financial_data.csv суммируем строки по id, потому что дубли отражают части оплаты
- в prolongations.csv удаляем дубли
- если для одного и того же (id, month) осталось несколько строк с разными менеджерами, выбираем менеджера из prolongations.csv, совпадающего с Account из financial_data.csv,
 если совпадения нет, берем первую запись и отдельно фиксируем конфликт


> В financial_data один и тот же проект может встречаться несколько раз. Поэтому сначала нужно собрать данные так, чтобы каждому id соответствовала одна строка. Таблица группируется по id, и для каждого проекта суммируются все значения по месячным колонкам. На выходе получается таблица fin_agg, где у каждого проекта одна строка и общая отгрузка по месяцам. Так как по одному проекту может быть несколько строк и в них может повторяться имя менеджера, код берёт самое часто встречающееся значение. Если по проекту имя менеджера отсутствует, возвращается None

In [88]:
fin_agg = fin.groupby('id', as_index=False)[month_cols].sum()
fin_account = fin.groupby('id')['Account'].agg(
    lambda s: s.dropna().astype(str).str.strip().mode().iat[0] if not s.dropna().empty else None
)

> Очищаем таблицу prolongations от дублей и не нужных значений

In [89]:
prol['AM'] = prol['AM'].fillna('без А/М').astype(str).str.strip().replace({'': 'без А/М'})
prol['end_month_ts'] = prol['month'].map(parse_month_label)
prol_initial_rows = len(prol)
prol_exact_dedup = prol.drop_duplicates()

> Разрешение конфликтов по id и month. Если для пары id + month строка только одна, значит конфликта нет. Эта строка просто добавляется в итоговый список. Если строк несколько, значит есть неоднозначность. Например, по одному проекту и одному месяцу в prolongations могут быть указаны разные менеджеры.

In [90]:
issues = []
resolved_rows = []

for (id_, month), grp in prol_exact_dedup.groupby(['id', 'month'], sort=False):
    if len(grp) == 1:
        resolved_rows.append(grp.iloc[0].to_dict())
    else:
        account_name = fin_account.get(id_, None)
        match = grp[grp['AM'] == account_name]
        chosen = match.iloc[0] if len(match) >= 1 else grp.iloc[0]
        resolved_rows.append(chosen.to_dict())
        issues.append({
            'id': int(id_),
            'month': month,
            'AM_options': ' | '.join(grp['AM'].tolist()),
            'chosen_AM': chosen['AM'],
            'financial_account': account_name
        })



> Получаем: 

In [91]:
prol_clean = pd.DataFrame(resolved_rows)

print('Исходных строк в prolongations:', prol_initial_rows)
print('После удаления дублей:', len(prol_exact_dedup))
print('После разрешения конфликтов:', len(prol_clean))
print('Конфликтов id и month:', len(issues))

issue_df = pd.DataFrame(issues)
display(issue_df)

Исходных строк в prolongations: 477
После удаления дублей: 474
После разрешения конфликтов: 473
Конфликтов id и month: 1


,id,month,AM_options,chosen_AM,financial_account
0,361,сентябрь 2023,Смирнова Ольга Владимировна | Попова Екатерина...,Попова Екатерина Николаевна,Попова Екатерина Николаевна


> Для проекта 361 (сентябрь 2023) в prolongations.csv присутствовали разные варианты ФИО ответственного аккаунт-менеджера, однако выбранное итоговое значение совпало с менеджером, указанным в financial_data.csv Попова Екатерина Николаевна

## Объединение источников

> После очистки присоединяем к каждой записи из prolongations помесячные суммы из financial_data


In [92]:
base = prol_clean[['id', 'AM', 'month', 'end_month_ts']].merge(fin_agg, on='id', how='left')

missing_ids = sorted(set(prol_clean['id']) - set(fin_agg['id']))
for c in month_cols:
    base[c] = base[c].fillna(0.0)

managers = sorted(base['AM'].unique().tolist())
targets = pd.date_range('2023-01-01', '2023-12-01', freq='MS')

print('Менеджеры:', managers)
print('ID без финансовых данных:', missing_ids)
display(base)


Менеджеры: ['Васильев Артем Александрович', 'Иванова Мария Сергеевна', 'Кузнецов Михаил Иванович', 'Михайлов Андрей Сергеевич', 'Петрова Анна Дмитриевна', 'Попова Екатерина Николаевна', 'Смирнова Ольга Владимировна', 'Соколова Анастасия Викторовна', 'Федорова Марина Васильевна', 'без А/М']
ID без финансовых данных: []


,id,AM,month,end_month_ts,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024
0,42,Васильев Артем Александрович,ноябрь 2022,2022-11-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,453,Васильев Артем Александрович,ноябрь 2022,2022-11-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,548,Михайлов Андрей Сергеевич,ноябрь 2022,2022-11-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,87,Соколова Анастасия Викторовна,ноябрь 2022,2022-11-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,429,Соколова Анастасия Викторовна,ноябрь 2022,2022-11-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
468,955,Смирнова Ольга Владимировна,декабрь 2023,2023-12-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
469,1004,без А/М,декабрь 2023,2023-12-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
470,281,Соколова Анастасия Викторовна,декабрь 2023,2023-12-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
471,785,Соколова Анастасия Викторовна,декабрь 2023,2023-12-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,700.0,700.0,0.0,0.0,0.0,0.0


## Расчет коэффициентов по месяцам



> Функция format_month_ts получает месяц в формате даты и превращает его в строку для отчёта, например май 2023. Она нужна чисто для красивого отображения месяца в итоговой таблице

In [93]:
def format_month_ts(ts):
    ts = pd.Timestamp(ts)
    return f"{months_title[ts.month - 1]} {ts.year}"

> calc_monthly считает помесячные показатели пролонгации. Параметр kind определяет, какой именно коэффициент считается: 
> - 1 - пролонгация в первый месяц
> - 2 - пролонгация во второй месяц.
> - rows - список, в который по ходу расчёта собираются строки будущей итоговой таблицы

> Функция проходит по всем месяцам отчёта из списка `targets`. Для каждого такого месяца определяется предыдущий месяц и месяц двумя шагами назад. После этого по словарю `col_by_ts` находятся названия колонок с отгрузками, соответствующие текущему и предыдущим месяцам.
Если `kind == 1`, функция считает пролонгацию в первый месяц. Для этого она берёт из таблицы `base` только те проекты, которые завершились в предыдущем месяце. Их отгрузка в последний месяц работы записывается в колонку `last_ship`, а отгрузка в текущем месяце в колонку `renew_ship`. Затем создаётся признак `renewed`, который показывает, был ли проект пролонгирован, то есть есть ли у него отгрузка в текущем месяце больше нуля. Если `kind != 1`, функция считает пролонгацию во второй месяц. Тогда в базу берутся проекты, завершившиеся двумя месяцами ранее. Для них сохраняется отгрузка в последний месяц работы, отгрузка в первый месяц после завершения и отгрузка во второй месяц. После этого из базы исключаются проекты, которые уже были пролонгированы в первый месяц, то есть имели отгрузку в `t-1`. На оставшихся проектах снова создаётся признак `renewed`, показывающий, вернулся ли проект во второй месяц.
Когда база для конкретного месяца уже сформирована, функция проходит по каждому менеджеру из списка `managers` и дополнительно считает общие показатели по всему отделу, используя строку `ИТОГО ОТДЕЛ`. Для каждого менеджера либо для всего отдела выбирается соответствующий срез данных. По этому срезу формируется одна строка результата. Под суммой базы понимается сумма отгрузки последнего месяца по всем проектам, вошедшим в расчёт, а под суммой пролонгаций сумма отгрузки в текущем месяце только по тем проектам, которые действительно продлились. Каждая такая строка добавляется в список `rows`.

> После того как функция проходит по всем месяцам и всем менеджерам, список строк превращается в датафрейм `out`. В нём дополнительно считаются две производные метрики. 
> - Первая это конверсия по проектам, то есть доля продлённых проектов от общего числа проектов в базе.
> - Вторая коэффициент пролонгации, то есть отношение суммы пролонгаций к сумме базы.

> В обоих случаях перед делением проверяется, что знаменатель больше нуля, чтобы избежать деления на ноль, если базы нет, ставится `NaN`.


In [94]:
targets

DatetimeIndex(['2023-01-01', '2023-02-01', '2023-03-01', '2023-04-01',
               '2023-05-01', '2023-06-01', '2023-07-01', '2023-08-01',
               '2023-09-01', '2023-10-01', '2023-11-01', '2023-12-01'],
              dtype='datetime64[ns]', freq='MS')

In [95]:
def calc_monthly(kind):
    rows = []

    for t in targets:
        prev1 = t - pd.DateOffset(months=1)
        prev2 = t - pd.DateOffset(months=2)

        col_t = col_by_ts[pd.Timestamp(t)]
        col_prev1 = col_by_ts[pd.Timestamp(prev1)]

        if kind == 1:
            g = base[base['end_month_ts'] == prev1].copy()
            g['last_ship'] = g[col_prev1]
            g['renew_ship'] = g[col_t]
            g['renewed'] = g['renew_ship'] > 0

        else:
            col_prev2 = col_by_ts[pd.Timestamp(prev2)]
            g = base[base['end_month_ts'] == prev2].copy()
            g['last_ship'] = g[col_prev2]
            g['ship_m1'] = g[col_prev1]
            g['renew_ship'] = g[col_t]

            # оставляем только проекты, которые не были пролонгированы в первый месяц
            g = g[g['ship_m1'] <= 0].copy()
            g['renewed'] = g['renew_ship'] > 0

        for am in managers + ['ИТОГО ОТДЕЛ']:
            sub = g if am == 'ИТОГО ОТДЕЛ' else g[g['AM'] == am]

            rows.append({
                'Месяц отчета': format_month_ts(t),
                'Менеджер': am,
                'Проектов в базе': int(len(sub)),
                'Продлено проектов': int(sub['renewed'].sum()) if len(sub) else 0,
                'Сумма базы, ₽': float(sub['last_ship'].sum()) if len(sub) else 0.0,
                'Сумма пролонгаций, ₽': float(sub.loc[sub['renewed'], 'renew_ship'].sum()) if len(sub) else 0.0,
            })

    out = pd.DataFrame(rows)
    out['Конверсия по проектам'] = np.where(
        out['Проектов в базе'] > 0,
        out['Продлено проектов'] / out['Проектов в базе'],
        np.nan
    )
    out['Коэффициент пролонгации'] = np.where(
        out['Сумма базы, ₽'] > 0,
        out['Сумма пролонгаций, ₽'] / out['Сумма базы, ₽'],
        np.nan
    )
    return out


> Получаем

In [96]:
m1_full = calc_monthly(kind=1)
m2_full = calc_monthly(kind=2)

display(m1_full.head(12))
display(m2_full.head(12))

,Месяц отчета,Менеджер,Проектов в базе,Продлено проектов,"Сумма базы, ₽","Сумма пролонгаций, ₽",Конверсия по проектам,Коэффициент пролонгации
0,Январь 2023,Васильев Артем Александрович,22,0,0.0,0.0,0.0,NaN
1,Январь 2023,Иванова Мария Сергеевна,16,0,0.0,0.0,0.0,NaN
2,Январь 2023,Кузнецов Михаил Иванович,1,0,0.0,0.0,0.0,NaN
3,Январь 2023,Михайлов Андрей Сергеевич,10,0,0.0,0.0,0.0,NaN
4,Январь 2023,Петрова Анна Дмитриевна,0,0,0.0,0.0,NaN,NaN
5,Январь 2023,Попова Екатерина Николаевна,6,0,0.0,0.0,0.0,NaN
6,Январь 2023,Смирнова Ольга Владимировна,3,0,0.0,0.0,0.0,NaN
7,Январь 2023,Соколова Анастасия Викторовна,9,0,0.0,0.0,0.0,NaN
8,Январь 2023,Федорова Марина Васильевна,0,0,0.0,0.0,NaN,NaN
9,Январь 2023,без А/М,0,0,0.0,0.0,NaN,NaN


,Месяц отчета,Менеджер,Проектов в базе,Продлено проектов,"Сумма базы, ₽","Сумма пролонгаций, ₽",Конверсия по проектам,Коэффициент пролонгации
0,Январь 2023,Васильев Артем Александрович,12,0,0.0,0.0,0.0,NaN
1,Январь 2023,Иванова Мария Сергеевна,1,0,0.0,0.0,0.0,NaN
2,Январь 2023,Кузнецов Михаил Иванович,0,0,0.0,0.0,NaN,NaN
3,Январь 2023,Михайлов Андрей Сергеевич,4,0,945.0,0.0,0.0,0.0
4,Январь 2023,Петрова Анна Дмитриевна,0,0,0.0,0.0,NaN,NaN
5,Январь 2023,Попова Екатерина Николаевна,2,0,0.0,0.0,0.0,NaN
6,Январь 2023,Смирнова Ольга Владимировна,0,0,0.0,0.0,NaN,NaN
7,Январь 2023,Соколова Анастасия Викторовна,5,0,0.0,0.0,0.0,NaN
8,Январь 2023,Федорова Марина Васильевна,0,0,0.0,0.0,NaN,NaN
9,Январь 2023,без А/М,0,0,0.0,0.0,NaN,NaN


## Итоги по менеджерам и по отделу

> Годовой коэффициент считаем как взвешенное отношение сумм:

> - годовой M1 = `сумма всех числителей M1 / сумма всех знаменателей M1`
> - годовой M2 = `сумма всех числителей M2 / сумма всех знаменателей M2`


In [97]:
annual_m1 = (m1_full.groupby('Менеджер', as_index=False)[['Проектов в базе', 'Продлено проектов', 'Сумма базы, ₽', 'Сумма пролонгаций, ₽']].sum())
annual_m1['Конверсия по проектам'] = annual_m1['Продлено проектов'] / annual_m1['Проектов в базе']
annual_m1['Коэффициент пролонгации'] = annual_m1['Сумма пролонгаций, ₽'] / annual_m1['Сумма базы, ₽']

display(annual_m1.sort_values('Коэффициент пролонгации', ascending=False))

,Менеджер,Проектов в базе,Продлено проектов,"Сумма базы, ₽","Сумма пролонгаций, ₽",Конверсия по проектам,Коэффициент пролонгации
0,Васильев Артем Александрович,107,0,0.0,0.0,0.0,NaN
1,ИТОГО ОТДЕЛ,380,0,0.0,0.0,0.0,NaN
2,Иванова Мария Сергеевна,47,0,0.0,0.0,0.0,NaN
3,Кузнецов Михаил Иванович,15,0,0.0,0.0,0.0,NaN
4,Михайлов Андрей Сергеевич,26,0,0.0,0.0,0.0,NaN
5,Петрова Анна Дмитриевна,1,0,0.0,0.0,0.0,NaN
6,Попова Екатерина Николаевна,61,0,0.0,0.0,0.0,NaN
7,Смирнова Ольга Владимировна,50,0,0.0,0.0,0.0,NaN
8,Соколова Анастасия Викторовна,72,0,0.0,0.0,0.0,NaN
9,Федорова Марина Васильевна,0,0,0.0,0.0,NaN,NaN


In [98]:
annual_m2 = (m2_full.groupby('Менеджер', as_index=False)[['Проектов в базе', 'Продлено проектов', 'Сумма базы, ₽', 'Сумма пролонгаций, ₽']].sum())
annual_m2['Конверсия по проектам'] = annual_m2['Продлено проектов'] / annual_m2['Проектов в базе']
annual_m2['Коэффициент пролонгации'] = annual_m2['Сумма пролонгаций, ₽'] / annual_m2['Сумма базы, ₽']

display(annual_m2.sort_values('Коэффициент пролонгации', ascending=False))

,Менеджер,Проектов в базе,Продлено проектов,"Сумма базы, ₽","Сумма пролонгаций, ₽",Конверсия по проектам,Коэффициент пролонгации
1,ИТОГО ОТДЕЛ,364,0,945.0,0.0,0.0,0.0
4,Михайлов Андрей Сергеевич,29,0,945.0,0.0,0.0,0.0
0,Васильев Артем Александрович,112,0,0.0,0.0,0.0,NaN
2,Иванова Мария Сергеевна,48,0,0.0,0.0,0.0,NaN
3,Кузнецов Михаил Иванович,9,0,0.0,0.0,0.0,NaN
5,Петрова Анна Дмитриевна,0,0,0.0,0.0,NaN,NaN
6,Попова Екатерина Николаевна,59,0,0.0,0.0,0.0,NaN
7,Смирнова Ольга Владимировна,38,0,0.0,0.0,0.0,NaN
8,Соколова Анастасия Викторовна,68,0,0.0,0.0,0.0,NaN
9,Федорова Марина Васильевна,0,0,0.0,0.0,NaN,NaN


In [ ]:
print(annual_m1[annual_m1['Менеджер'] == 'ИТОГО ОТДЕЛ'].iloc[0], '\n')
print(annual_m2[annual_m2['Менеджер'] == 'ИТОГО ОТДЕЛ'].iloc[0])

Менеджер                   ИТОГО ОТДЕЛ
Проектов в базе                    380
Продлено проектов                    0
Сумма базы, ₽                      0.0
Сумма пролонгаций, ₽               0.0
Конверсия по проектам              0.0
Коэффициент пролонгации            NaN
Name: 1, dtype: object 

Менеджер                   ИТОГО ОТДЕЛ
Проектов в базе                    364
Продлено проектов                    0
Сумма базы, ₽                    945.0
Сумма пролонгаций, ₽               0.0
Конверсия по проектам              0.0
Коэффициент пролонгации            0.0
Name: 1, dtype: object
